# 🚦 Smart Traffic Annotation System

**AI-powered traffic video annotation with real-time vehicle detection, tracking & analytics**

| Feature | Technology |
|---|---|
| **Detection** | YOLOv8 (Ultralytics) |
| **Tracking** | DeepSORT |
| **Backend** | FastAPI + SQLite |
| **Frontend** | HTML5 Canvas + JS |
| **Analytics** | Counting, Speed, Lane Analysis, Safety (TTC) |
| **Export** | COCO, YOLO, VOC, CSV, JSON |

> **Run cells in order. Enable GPU:** `Runtime > Change runtime type > T4 GPU`


## 📋 Step 1: Check GPU & Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
!pip install -q fastapi==0.104.1 "uvicorn[standard]==0.24.0" python-multipart==0.0.6 \
    opencv-python==4.8.1.78 numpy==1.24.3 pillow==10.1.0 \
    sqlalchemy==2.0.23 pydantic==2.5.0 python-dotenv==1.0.0 \
    "ultralytics>=8.0.0" "deep-sort-realtime>=1.3.2" pyngrok
print('All dependencies installed!')


## 🏗 Step 2: Clone Repository & Setup

In [ ]:
import os, shutil

# Clone the repository
!git clone https://github.com/Dakshbumb/traffic-annotation-system.git /content/traffic-app 2>/dev/null || echo 'Already cloned'

os.chdir('/content/traffic-app/backend')
print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))


## 🤖 Step 3: Download YOLOv8 Model

In [ ]:
os.chdir('/content/traffic-app/backend')
from ultralytics import YOLO
print('Downloading YOLOv8m model...')
model = YOLO('yolov8m.pt')
print(f'Model ready! File exists: {os.path.exists("yolov8m.pt")}')


## 🌐 Step 4: Setup ngrok Tunnel

1. Sign up free at [ngrok.com](https://ngrok.com)
2. Copy your authtoken from [dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Paste below


In [ ]:
NGROK_TOKEN = input('Enter ngrok authtoken: ')
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('ngrok configured!')


## 🚀 Step 5: Start the Server

In [ ]:
import subprocess, time, torch
from pyngrok import ngrok

os.chdir('/content/traffic-app/backend')

# Start FastAPI
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)

# Create tunnel
public_url = ngrok.connect(8000)

print('=' * 60)
print('SMART TRAFFIC ANNOTATION SYSTEM')
print('=' * 60)
print(f'\nPublic URL: {public_url}')
print(f'API Docs:   {public_url}/docs')
print(f'GPU: {"CUDA" if torch.cuda.is_available() else "CPU"}')
print('\nUsage:')
print('  1. Click Public URL to open web interface')
print('  2. Upload a traffic video (MP4/AVI/MOV)')
print('  3. Click Auto-Label for YOLOv8+DeepSORT detection')
print('  4. Use Edit/Analytics/Lanes/Safety modes')
print('  5. Export in COCO/YOLO/VOC/CSV format')
print('\nKeep this cell running!')
print('=' * 60)


In [ ]:
# Keep server running - press STOP to shut down
try:
    while True:
        time.sleep(60)
        if server.poll() is not None:
            print('Server stopped! Logs:')
            print(server.stderr.read().decode()[-3000:])
            break
except KeyboardInterrupt:
    server.terminate()
    ngrok.kill()
    print('Shut down complete.')


## 📹 (Optional) Upload a Video

In [ ]:
from google.colab import files
uploaded = files.upload()
for fn in uploaded:
    with open(f'/content/traffic-app/backend/uploads/{fn}', 'wb') as f:
        f.write(uploaded[fn])
    print(f'Uploaded: {fn} ({len(uploaded[fn])/1024/1024:.1f} MB)')


## 📋 (Optional) Server Logs

In [ ]:
if server.poll() is None:
    print('Server is running')
else:
    print('Server stopped')
    print(server.stderr.read().decode()[-2000:])
